# NeuroFlex CPU–GPU validation
This notebook compares identical batched joint-angle mathematics on CPU and CUDA. It reports compute-only GPU time separately from host-to-device transfer-inclusive time. It validates engineering execution, not clinical accuracy. Run once with **Runtime → Change runtime type → T4 GPU** and once with **CPU**.

In [ ]:
import json, platform, statistics, time
import numpy as np
import torch
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device('cuda' if CUDA_AVAILABLE else 'cpu')
print({'platform': platform.platform(), 'torch': torch.__version__, 'cuda_available': CUDA_AVAILABLE, 'selected': str(DEVICE), 'gpu': torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None, 'cuda_runtime': torch.version.cuda})

## Fallback decision used by NeuroFlex
`auto` and `cuda` select CUDA only when PyTorch is installed and `torch.cuda.is_available()` returns true. Otherwise the engine returns a NumPy CPU implementation and records a fallback reason. Explicit `cpu` always stays on CPU.

In [ ]:
def resolve_device(preference='auto'):
    if preference not in {'auto', 'cpu', 'cuda'}: raise ValueError(preference)
    available = torch.cuda.is_available()
    if preference != 'cpu' and available: return {'requested': preference, 'selected': 'cuda', 'reason': None}
    reason = 'CUDA requested but unavailable; using CPU' if preference == 'cuda' else ('CUDA unavailable; using CPU' if preference == 'auto' and not available else None)
    return {'requested': preference, 'selected': 'cpu', 'reason': reason}
for preference in ('auto', 'cpu', 'cuda'): print(resolve_device(preference))

In [ ]:
def numpy_angles(points):
    left, right = points[:, 0] - points[:, 1], points[:, 2] - points[:, 1]
    denominator = np.linalg.norm(left, axis=1) * np.linalg.norm(right, axis=1)
    cosine = np.sum(left * right, axis=1) / denominator
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

def torch_angles(points):
    left, right = points[:, 0] - points[:, 1], points[:, 2] - points[:, 1]
    denominator = torch.linalg.vector_norm(left, dim=1) * torch.linalg.vector_norm(right, dim=1)
    cosine = torch.sum(left * right, dim=1) / denominator
    return torch.rad2deg(torch.acos(torch.clamp(cosine, -1.0, 1.0)))

def percentile(values, p): return float(np.percentile(np.asarray(values), p))

def benchmark(batch_size, iterations=200, warmup=30, seed=20261030):
    rng = np.random.default_rng(seed + batch_size)
    host = rng.normal(size=(batch_size, 3, 3)).astype(np.float32)
    for _ in range(warmup): numpy_angles(host)
    cpu_times = []
    for _ in range(iterations):
        start = time.perf_counter_ns(); numpy_angles(host)
        cpu_times.append((time.perf_counter_ns() - start) / 1e6)
    result = {'batch': batch_size, 'cpu_median_ms': statistics.median(cpu_times), 'cpu_p95_ms': percentile(cpu_times, 95)}
    if CUDA_AVAILABLE:
        resident = torch.as_tensor(host, device='cuda')
        for _ in range(warmup): torch_angles(resident)
        torch.cuda.synchronize(); compute_times, end_to_end_times = [], []
        for _ in range(iterations):
            start = time.perf_counter_ns(); torch_angles(resident); torch.cuda.synchronize()
            compute_times.append((time.perf_counter_ns() - start) / 1e6)
            start = time.perf_counter_ns(); transferred = torch.as_tensor(host, device='cuda'); output = torch_angles(transferred); output.cpu(); torch.cuda.synchronize()
            end_to_end_times.append((time.perf_counter_ns() - start) / 1e6)
        result.update({'gpu_compute_median_ms': statistics.median(compute_times), 'gpu_compute_p95_ms': percentile(compute_times, 95), 'gpu_transfer_inclusive_median_ms': statistics.median(end_to_end_times), 'gpu_transfer_inclusive_p95_ms': percentile(end_to_end_times, 95), 'compute_speedup': statistics.median(cpu_times) / statistics.median(compute_times), 'end_to_end_speedup': statistics.median(cpu_times) / statistics.median(end_to_end_times)})
        assert np.allclose(numpy_angles(host), torch_angles(resident).cpu().numpy(), atol=1e-3)
    return result

RESULTS = [benchmark(size) for size in (1, 32, 256, 2048, 10000)]
print(json.dumps(RESULTS, indent=2))

## Interpretation
Small real-time batches can be faster on CPU because CUDA kernel launch and transfer overhead dominate. GPU acceleration becomes useful as batch size grows or when tensors remain resident on the GPU. NeuroFlex therefore selects CPU safely when CUDA is absent and does not claim that every task belongs on the GPU.

In [ ]:
from google.colab import files
payload = {'environment': {'torch': torch.__version__, 'cuda_available': CUDA_AVAILABLE, 'device': str(DEVICE), 'gpu': torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None}, 'results': RESULTS}
name = 'neuroflex_cuda_benchmark.json' if CUDA_AVAILABLE else 'neuroflex_cpu_colab_benchmark.json'
with open(name, 'w') as handle: json.dump(payload, handle, indent=2)
files.download(name)